# Linear Text Classifiers: Generative vs. Discriminative

Naive Bayes (NB) and logistic regression (LR) draw the same kind of boundary through the space of word counts — a
hyperplane — and disagree only about how to place it. NB fits a model of the words given the class and reads the
boundary off it; LR fits the boundary directly. This notebook makes five points:
- NB's posterior is a softmax of a linear function of the counts, so the two classifiers share a hypothesis class
  (Prop 1), and they differ in which moments of the data they match (Thm 1);
- NB's parameters converge with $O(\log V)$ documents when every feature's rate is bounded away from 0 (Lemma 1,
  Thm 2), but a multinomial vocabulary's rates are at most $1/V$, and Zipf makes the tail far rarer (Thm 3);
- burstiness keeps LR well specified while NB overcounts repeated words, which produces the generative–discriminative
  crossover in worlds whose Bayes error we know (Prop 2);
- with more words than documents the classes are separable, the logistic MLE does not exist, and the regularization
  path turns toward the max-margin separator (Thm 4); NB features act as a per-word prior (Prop 3);
- in the finance corpus, tone is an XOR of aspect valence and direction ("costs rose" is bad news), which no unigram
  model represents and bigrams do (Prop 4).

Every number below comes from `linear_text_classifiers.py`, which asserts each claim.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "linear_text_classifiers.py").exists()
                       else here / "notebooks" / "linear-text-classifiers"))
import linear_text_classifiers as ltc

X, y, voc = ltc.corpus_binary()
print("polar training documents", X.shape[0], " vocabulary", X.shape[1], " informative words |S| =", len(ltc.informative_words()))

## 1. One hypothesis class

A document becomes its **count vector** $x \in \mathbb N^V$, $x_j$ the number of times word $j$ occurs (Def 1); its
length is $L(x) = \sum_j x_j$. Multinomial NB says $P(x \mid y = k) \propto \prod_j \theta_{kj}^{x_j}$, so

$$\log P(y = k \mid x) = \operatorname{softmax}_k(Wx + b), \qquad W_{kj} = \log\theta_{kj},\; b_k = \log \pi_k .$$

Bernoulli NB is the same form on binary presence, with $W_{kj} = \operatorname{logit}\theta_{kj}$ and an extra bias
$\sum_j \log(1 - \theta_{kj})$ (Prop 1(a)). Conversely, any $\operatorname{softmax}(Wx + b)$ is an NB posterior on
documents of one fixed length (normalize each row; the normalizers become $L \log Z_k$, a bias) — but not across
lengths (Prop 1(b)). We check both, and that the closed form matches scikit-learn.

In [ ]:
ltc.test_nb_is_softmax_linear()
ltc.test_fixed_length_families_coincide()
ltc.test_nb_matches_scikit_learn()
print("Prop 1 checks pass")

## 2. Two estimating equations

Penalized softmax LR minimizes $\frac1n\sum_i -\log p_{i,y_i} + \frac\lambda2\|W\|^2$. Its Hessian is
$\frac1n\sum_i (\operatorname{diag} p_i - p_i p_i^\top) \otimes \tilde x_i \tilde x_i^\top + \lambda I$ with
$\tilde x = (x, 1)$ — positive semidefinite, with a null space of dimension $V + 1$ at $\lambda = 0$ (adding one
vector to every class's weights changes nothing) and $1$ at $\lambda > 0$. Setting the gradient to zero gives
**Theorem 1**'s equations

$$\sum_i (y_{ik} - p_{ik})\,x_{ij} = n\lambda W_{kj}, \qquad \sum_i p_{ik} = n_k .$$

LR matches every class × word co-count (exactly when $\lambda = 0$) and every class total. NB matches class-conditional
word totals instead, and on the finance corpus misses its own class totals.

In [ ]:
ltc.test_gradient_and_hessian()
ltc.test_lr_matches_scikit_learn()
ltc.test_estimating_equations()
f_lr, f_nb = ltc.fin_fit(ltc.FinConfig("lr", False)), ltc.fin_fit(ltc.FinConfig("nb", False))
Xf = ltc.fin_features(False)[0]; yf = ltc.fin_split()[1]
for name, f, lam in (("LR", f_lr, f_lr["hp"]), ("NB", f_nb, 0.0)):
    R, t = ltc.estimating_residuals(Xf, yf, f["W"], f["b"], lam)
    print(f"{name}: largest co-count residual {np.abs(R).max():.3g}, largest class-total residual {np.abs(t).max():.3g}")

## 3. How fast: Ng and Jordan's generative bound

Ng and Jordan (2002) take $V$ binary features with every rate in $[\rho_0, 1 - \rho_0]$. **Lemma 1:** with
$m_0 = \lceil \ln(4V/\delta)/(2\varepsilon^2) \rceil$ documents per class, every estimated rate is within
$\varepsilon$ with probability $\ge 1 - \delta$ (Hoeffding plus a union bound — the $\log V$). **Theorem 2:** on that
event each of the discriminant's $2V + 2$ log terms moves by at most $2\varepsilon/\rho_0$, so
$|\hat l(x) - l_\infty(x)| \le \tau V$ with $\tau V = 4(V + 1)\varepsilon/\rho_0$, and NB can lose only documents
that its population version classifies within that margin: $\varepsilon(\hat h) \le \varepsilon(h_\infty) + G(\tau)$.

We check both in a Bernoulli world built to meet the hypotheses ($V = 200$, $\rho_0 = 0.1$, $\varepsilon = 0.05$,
$\delta = 0.2$). The bound is sufficient and loose: it adds the $V$ errors as if they all pointed the same way.

In [ ]:
ltc.test_lemma1_concentration()
ltc.test_thm2_margin_bound()
nj = ltc.ng_jordan_check()
for k in ("m0", "fail_rate", "dev_quantile", "max_gap", "tauV", "G", "median_abs_l", "m0_for_unit_tau"):
    print(f"{k:18} {nj[k]}")

## 4. Multinomial counts and the Zipf tail

A multinomial's rates sum to 1, so some word has $\theta_j \le 1/V$: Ng and Jordan's $\rho_0$ cannot hold uniformly.
**Theorem 3:** (a) $N \ge 3\ln(2V/\delta)/(\varepsilon^2\theta_{\min})$ tokens per class put every $\hat\theta_j$ within
a factor $1 \pm \varepsilon$ (multiplicative Chernoff); then each log-rate moves by at most $2\varepsilon$ and a
length-$L$ document's discriminant by at most $4\varepsilon L$. (b) Fewer than about $1/\theta_{\min}$ tokens leave the
rarest word unseen with constant probability, so uniform relative accuracy costs at least order $V$ tokens — and
$V^s \log V$ under Zipf($s$), Topic 1's law. In the long-tailed world below the bound asks for far more text than
4,096 documents supply, and the validated smoothing strength $\alpha$ grows with $n$.

In [ ]:
ltc.test_thm3_relative_accuracy_costs_one_over_theta_min()
zt = ltc.zipf_world_tokens()
for k in ("V", "theta_min", "tokens_needed", "tokens_supplied", "unseen_words", "uniform_tokens_needed", "zipf_tokens_needed"):
    print(f"{k:22} {zt[k]:.6g}")
print("share of test tokens from words seen < 5 times:", {n: round(v, 4) for n, v in zt["rare_share"].items()})
c = ltc.learning_curve(ltc.WorldConfig(ltc.WORLD_NOISE[-1], False))
print("validated alpha (median over reps) by n:", {n: float(a) for n, a in zip(ltc.CURVE_N, np.median(c["nb_alpha"], 0))})

## 5. Burstiness and the crossover

Real words come in bursts. Let word $j$'s every occurrence repeat $r_j$ times. **Proposition 2:** (a) the Bayes rule
on the observed counts is still linear, $\sum_j (x_j/r_j)\log(\theta_{1j}/\theta_{0j})$, so LR stays well specified;
(b) NB fit to infinitely many such documents estimates $\tilde\theta_k \propto r \odot \theta_k$ and weights each
observed count by $\log(\theta_{1j}/\theta_{0j})$ plus a constant — $r_j$ times too much on bursty words. Its
asymptotic error exceeds the Bayes error, while LR's does not (Ng and Jordan's Prop 1).

The learning curves: tuned NB and tuned LR (each hyperparameter picked on a fresh validation draw), four nested
training draws per world, test accuracy on 3,000 documents.

In [ ]:
ltc.test_bursty_world_keeps_lr_well_specified()
ltc.test_generative_discriminative_regimes()
for nw in ltc.WORLD_NOISE:
    for b in (False, True):
        c = ltc.learning_curve(ltc.WorldConfig(nw, b))
        print(f"V = {c['V']:5} {'bursty ' if b else 'NB-true'}  Bayes {c['bayes']:.3f}  NB-asymptote {c['nb_inf']:.3f}  crossover {ltc.crossover_n(c)}")
        print("   n  ", " ".join(f"{n:6}" for n in ltc.CURVE_N))
        print("   NB ", " ".join(f"{a:6.3f}" for a in c["nb"].mean(0)))
        print("   LR ", " ".join(f"{a:6.3f}" for a in c["lr"].mean(0)))

## 6. Separation

With more words than documents the training set is almost always separable: if the $n$ count vectors are linearly
independent, $Xw = s$ has a solution for every sign vector $s$. **Theorem 4:** (a) on separable data the logistic loss
has infimum 0 and no minimizer, since along $t\hat w$ it falls to 0 and its gradient never vanishes; (b) the 200 polar
tone documents ($V = 2{,}738$) have full row rank; (c) the direction of the $L_2$ path, $\hat w_\lambda/\|\hat w_\lambda\|$,
converges to the hard-margin SVM direction as $\lambda \to 0$ (Rosset, Zhu, and Hastie 2004, Thm 2.1), while
$\|\hat w_\lambda\|$ grows like $\ln(1/\lambda)/\gamma^*$. Gradient descent from zero turns the same way, at Soudry et
al.'s (2018) logarithmic pace.

In [ ]:
ltc.test_separable_means_no_mle()
ltc.test_l2_path_turns_toward_max_margin()
sep = ltc.separation()
print("SVM normalized margin", round(sep["svm_margin"], 6))
for p in sep["path"]:
    print(f"lambda {p['lam']:.0e}  |w| {p['norm']:7.3f}  cos {p['cos']:.6f}  margin {p['margin']:.6f}")
for g in sep["gd"]:
    print(f"GD t {g['t']:6}  |w| {g['norm']:6.3f}  cos {g['cos']:.6f}  loss {g['loss']:.5f}")

## 7. Naive Bayes features as a prior

Wang and Manning's log-count ratio $r = \log\big((p/\|p\|_1)/(q/\|q\|_1)\big)$, $p = \alpha + \sum_{y=1} f$,
$q = \alpha + \sum_{y=0} f$, is NB's weight vector. **Proposition 3:** (a) LR on $r \odot x$ with penalty
$\lambda\|v\|^2$ is LR on $x$ with the per-word penalty $\lambda \sum_j w_j^2/r_j^2$ — weak NB evidence becomes a strong
prior toward 0; (b) $|r_j| \le \ln\big((c_j + \alpha)/\alpha\big) + \big|\ln\frac{N_1 + \alpha V}{N_0 + \alpha V}\big|$, so a
word seen once can carry a log ratio of at most $\ln(1 + 1/\alpha)$ plus a constant: the funnel.

In [ ]:
ltc.test_nb_weighting_is_a_feature_ridge()
ltc.test_smoothing_bounds_log_count_ratios()
fu = ltc.funnel()
once = fu["count"] == 1
for a in ltc.FUNNEL_ALPHAS:
    print(f"alpha {a:5}: largest |r| among words seen once {np.abs(fu['r'][a][once]).max():.3f}   bound {fu['bound'][a][once].max():.3f}")

## 8. Finance case study: tone

The synthetic filings carry a document tone ($-1, 0, +1$) expressed through 5–8 aspect sentences. **Proposition 4:**
"revenue rose" and "costs fell" are good news, "revenue fell" and "costs rose" bad — four one-clause documents no
hyperplane in unigram counts separates, and bigram indicators do. On the corpus, LR leaves the direction words
near zero and puts the sign on the bigrams. The full-text Bayes accuracy (a reader who parses every aspect sentence)
is the ceiling; NB, LR, and NB-weighted LR on unigrams and bigrams fall short of it, NB is badly overconfident, and
LR on bigrams — more features than documents — is overconfident too.

In [ ]:
ltc.test_aspect_xor()
ltc.test_bayes_ceiling()
ltc.test_finance_tone()
ltc.test_label_noise()
ltc.test_finance_has_no_crossover()
print("full-text Bayes accuracy", round(ltc.bayes_ceiling(), 6))
pc = ltc.fin_prefix_curve()
print("training prefix n", pc["n"], " NB", [round(a, 3) for a in pc["nb"]], " LR", [round(a, 3) for a in pc["lr"]])
for m in ltc.FIN_METHODS:
    for bg in (False, True):
        cal = ltc.fin_calibration(ltc.FinConfig(m, bg))
        print(f"{m:3} {'bigram ' if bg else 'unigram'}  acc {cal['acc']:.3f}  conf {cal['conf']:.3f}  ECE15 {cal['ece']:.3f}")
for f in ltc.XOR_FEATURES:
    print(f"{f:12} uni {ltc.fin_weight(ltc.FinConfig('lr', False), f):7.3f}   bi {ltc.fin_weight(ltc.FinConfig('lr', True), f):7.3f}")

## 9. The lab payload

Every number the page's lab prints is a value in `viz_payload()`, rounded in Python; the JSON file in
`src/components/viz/data/` is pinned to it.

In [ ]:
ltc.test_viz_payload()
ltc.test_viz_mirror_matches_payload()
p = ltc.viz_payload()
print({k: list(v) if isinstance(v, dict) else v for k, v in p.items() if k != "funnel"})

## Where this sits

Topic 18 replaces the learned weights with a dictionary (Loughran and McDonald) and asks what a lexicon loses;
Topic 21 lifts the same log-linear model to sequences (the linear-chain CRF); Topic 28 asks whether a 0.03 accuracy
gap between two of these classifiers is real. formalML's `naive-bayes` covers the Gaussian case and conjugate
smoothing; formalStatistics' `generalized-linear-models` covers binary logistic regression's asymptotics.